# 说明

## 功能说明
这个代码示例演示了**如何使用AI代理自动化处理差旅费用报销流程**，特别展示了从收据图片中提取费用信息并生成专业报销邮件的完整工作流。这是生产级AI应用的一个典型示例，展示了AI Agent在真实业务场景中的应用。

### 核心演示内容
1. **多模态处理能力**
    - 从收据图片中提取文本和费用信息（OCR功能）
    - 同时处理图像和文本输入
2. **代理协作机制**
    - 创建两个专业代理协同工作：
        - OCR代理：专门负责从图片中提取费用数据
        - 邮件代理：专门负责生成专业报销邮件
    - 代理间自动传递处理结果
3. **结构化数据处理**
    - 将非结构化的收据信息转换为结构化的费用数据
    - 使用Pydantic模型确保数据格式正确
4. **生产级应用流程**
    - 模拟真实报销场景的完整工作流
    - 处理实际业务需求（日期格式化、金额提取等）

注意：我在运行的时候有时候并没有完全严格按照提示词描述的格式输出，导致无法识别账单，邮件生成有误

# 报销分析

本笔记本展示了如何创建使用插件的代理来处理本地收据图像中的差旅费用，生成报销邮件，并通过饼图可视化费用数据。代理会根据任务上下文动态选择功能。

步骤：
1. OCR代理处理本地收据图像并提取差旅费用数据。
2. 邮件代理生成报销邮件。

### 差旅费用场景示例：
假设你是一名员工，为了参加另一座城市的商务会议而出差。你的公司有一项政策，会报销所有合理的差旅相关费用。以下是可能的差旅费用明细：
- 交通费用：
往返于家乡城市和目的地城市的机票费用。  
往返机场的出租车或网约车费用。  
目的地城市的本地交通费用（如公共交通、租车或出租车）。

- 住宿费用：
在会议场地附近的中档商务酒店住宿三晚的费用。

- 餐饮费用：
根据公司每日津贴政策，涵盖早餐、午餐和晚餐的每日餐饮补贴。

- 杂项费用：
机场停车费。  
酒店的网络使用费。  
小费或其他小额服务费用。

- 文档：
你需要提交所有收据（机票、出租车、酒店、餐饮等）以及一份完整的报销报告以申请报销。


## 导入所需库

导入笔记本所需的库和模块。


In [ ]:
# 导入必要的库 - 就像做饭前准备好所有食材
import os  # 操作系统相关功能，如读取文件
from dotenv import load_dotenv  # 用于加载.env文件中的敏感信息（如API密钥）
from azure.ai.inference import ChatCompletionsClient  # Azure AI服务客户端
from azure.core.credentials import AzureKeyCredential  # Azure认证凭证
from semantic_kernel.kernel import Kernel  # Semantic Kernel的核心类
from semantic_kernel.agents import AgentGroupChat  # 用于创建多代理协作
from openai import AsyncOpenAI  # 异步调用OpenAI API的客户端
from semantic_kernel.agents import ChatCompletionAgent  # 基于聊天的AI代理
from semantic_kernel.contents.utils.author_role import AuthorRole  # 消息角色定义
from semantic_kernel.agents.strategies import SequentialSelectionStrategy, DefaultTerminationStrategy  # 代理协作策略
from semantic_kernel.contents.chat_message_content import ChatMessageContent  # 聊天消息内容
from semantic_kernel.contents import ImageContent, TextContent  # 图像和文本内容类型
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion, OpenAIChatPromptExecutionSettings  # OpenAI连接器
from semantic_kernel.functions import kernel_function, KernelArguments  # 定义可调用函数的工具
from pydantic import BaseModel, Field  # 用于定义结构化数据模型
from typing import List  # 类型提示工具
from azure.ai.inference.models import SystemMessage, UserMessage, TextContentItem, ImageContentItem, ImageUrl, ImageDetailLevel  # Azure模型相关类


load_dotenv()

True

In [ ]:
def _create_kernel_with_chat_completion(service_id: str) -> Kernel:
    """
    创建并配置Semantic Kernel实例，连接到AI模型服务
    
    参数:
        service_id: 服务标识符，用于区分不同模型
    
    返回:
        Kernel: 配置好的Kernel实例
    """
     # 创建一个"厨房"（Kernel），所有AI处理都在这里进行
    kernel = Kernel()
   
    # 创建一个"厨师"（AsyncOpenAI客户端），负责与AI模型沟通
    client = AsyncOpenAI(
        api_key=os.environ["GITHUB_TOKEN"], 
        base_url="https://models.inference.ai.azure.com/"
    )
    
    # 添加一个"小型厨师"（gpt-4o-mini模型）
    kernel.add_service(
        OpenAIChatCompletion(
            ai_model_id="gpt-4o-mini",  # 使用的小型AI模型
            async_client=client,  # 使用上面创建的客户端
            service_id="open_ai"  # 服务标识
        )
    )
    
    # 添加一个"大型厨师"（gpt-4o模型）
    kernel.add_service(
        OpenAIChatCompletion(
            ai_model_id="gpt-4o",  # 使用的大型AI模型
            async_client=client,  # 使用相同的客户端
            service_id="gpt-4o"  # 服务标识
        )
    )

    return kernel

## 定义费用模型

创建一个 Pydantic 模型用于表示单项费用，并创建一个 ExpenseFormatter 类，将用户查询转换为结构化的费用数据。

每项费用将以以下格式表示：
`{'date': '07-Mar-2025', 'description': '飞往目的地的航班', 'amount': 675.99, 'category': '交通'}`


In [ ]:
# 定义费用数据结构 - 就像设计一个标准的报销单模板
class Expense(BaseModel):
    """
    表示单个费用项的数据模型
    所有费用必须按照这个格式组织
    """
    # 日期字段 - 必须按照"dd-MMM-yyyy"格式（如"07-Mar-2025"）
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    # 描述字段 - 费用的简要说明
    description: str = Field(..., description="Expense description")
    # 金额字段 - 必须是数字（如675.99）
    amount: float = Field(..., description="Expense amount")
    # 类别字段 - 必须是预定义的类别之一
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    """
    用于将原始查询转换为结构化费用数据的工具
    """
    # 原始查询 - 未经处理的费用数据
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        
        将原始查询解析为费用对象列表
        预期格式: "日期|描述|金额|类别"，用分号分隔多个费用
        返回:
            List[Expense]: 解析后的费用对象列表
        """
        expense_list = []  # 创建一个空列表，用于存放解析后的费用
        
        # 将原始查询按分号分割成多个费用项
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():  # 跳过空项
                # 将每个费用项按竖线分割成四部分
                parts = expense_str.strip().split("|")
                if len(parts) == 4:  # 确保有四个部分（日期、描述、金额、类别）
                    date, description, amount, category = parts
                    try:
                        # 创建一个Expense对象并添加到列表
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),  # 将金额字符串转为浮点数
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## 定义代理 - 生成邮件

创建一个代理类，用于生成提交报销申请的邮件。  
- 该代理使用 `kernel_function` 装饰器定义一个函数，用于生成提交报销申请的邮件。  
- 它会计算费用的总金额，并将详细信息格式化为邮件正文。  


In [ ]:
# 定义邮件生成代理 - 就像一个专门写报销邮件的秘书
class ExpenseEmailAgent:

    # 生成提交报销申请的邮件
    @kernel_function(description="Generate an email to submit an expense claim to the Finance Team")
    async def generate_expense_email(expenses):
        """
        生成专业的报销申请邮件
        
        参数:
            expenses: 费用列表，格式为[{'date': ..., 'description': ..., 'amount': ..., 'category': ...}, ...]
        
        返回:
            str: 格式化的报销邮件内容
        """
        # 计算所有费用的总金额
        total_amount = sum(expense['amount'] for expense in expenses)
        
        # 开始构建邮件内容
        email_body = "Dear Finance Team,\n\n"
        email_body += "Please find below the details of my expense claim:\n\n"
        # 为每项费用添加详细信息
        for expense in expenses:
            email_body += f"- {expense['description']}: ${expense['amount']}\n"
        
        # 添加总金额
        email_body += f"\nTotal Amount: ${total_amount}\n\n"
        # 添加标准结尾
        email_body += "Receipts for all expenses are attached for your reference.\n\n"
        email_body += "Thank you,\n[Your Name]"
        return email_body

# 从收据图片中提取差旅费用的代理

创建一个代理类，用于从收据图片中提取差旅费用。
- 该代理使用 `kernel_function` 装饰器定义一个函数，用于从收据图片中提取差旅费用。
- 使用 OCR（光学字符识别）将收据图片转换为文本，并提取相关信息，例如日期、描述、金额和类别。


In [ ]:
# 定义OCR代理插件 - 就像一个专门看收据的会计
class OCRAgentPlugin:
    def __init__(self):
        """初始化OCR代理"""
        # 创建与Azure AI服务的连接
        self.client = ChatCompletionsClient(
            endpoint="https://models.inference.ai.azure.com/",
            credential=AzureKeyCredential(os.environ.get("GITHUB_TOKEN")),
        )
        self.model_name = "gpt-4o"

    # 使用gpt-4o模型从receipt.jpg中提取结构化差旅费用数据
    @kernel_function(description="Extract structured travel expense data from receipt.jpg using gpt-4o-model")
    def extract_text(self, image_path: str = "receipt.jpg") -> str:
        """
        从收据图片中提取结构化费用数据
        
        参数:
            image_path: 收据图片的路径，默认为"receipt.jpg"
        
        返回:
            str: 提取的结构化费用数据
        """
        try:
            # 加载图片并准备发送给AI模型
            image_url_str = str(ImageUrl.load(
                image_file=image_path, 
                image_format="jpg", 
                detail=ImageDetailLevel.HIGH  # 高细节级别，确保准确提取
            ))
        
            # 准备AI模型的指示语（告诉AI要做什么）
            # 您是一位专业的 OCR 助手，擅长从收据图像中提取结构化数据。
            # 分析提供的收据图像并提取与旅行相关的费用详细信息，格式如下：
            # ‘日期|描述|金额|类别’以分号分隔。
            # 请遵循以下规则：
            # - 日期：将日期（例如，‘4/4/22’）转换为‘dd-MMM-yyyy’（例如，‘04-Apr-2022’）。
            # - 描述：提取项目名称（例如，‘Carlson's Drylawn’、‘Peigs transaction Probiotics’）。
            # - 金额：使用数值（例如，‘$4.50’ 中的‘4.50’或‘4.50 美元’）。
            # - 类别：根据上下文推断（例如，‘餐饮’代表食物，‘交通’代表旅行，‘住宿’代表住宿，‘杂项’代表其他）。
            # 忽略总计、小计或服务费，除非它们是分项费用。
            # 如果没有发现任何费用，则返回‘未检测到任何费用’。
            # 仅返回结构化数据，不返回附加文本。
            prompt = (
                "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
                "Analyze the provided receipt image and extract travel-related expense details in the format: "
                "'date|description|amount|category' separated by semicolons. "
                "Follow these rules: "
                "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
                "- Description: Extract item names (e.g., 'Carlson's Drylawn', 'Peigs transaction Probiotics'). "
                "- Amount: Use numeric values (e.g., '4.50' from '$4.50' or '4.50 dollars'). "
                "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, 'Accommodation' for lodging, 'Miscellaneous' otherwise). "
                "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
                "If no expenses are found, return 'No expenses detected'. "
                "Return only the structured data, no additional text."
            )
            # 调用AI模型处理图片
            response = self.client.complete(
                messages=[
                    # 系统消息：告诉AI它的角色和任务
                    SystemMessage(content=prompt),
                    # 用户消息：包含图片和简短说明
                    UserMessage(content=[
                        TextContentItem(text="Extract travel expenses from this receipt image."),
                        ImageContentItem(image_url=ImageUrl(url=image_url_str))
                    ])
                ],
                model=self.model_name,
                temperature=0.1,
                max_tokens=2048
            )
            # 获取AI模型的响应内容
            extracted_text = response.choices[0].message.content
            return extracted_text
        except Exception as e:
            error_msg = f"[LOG] OCR Plugin: Error processing image: {str(e)}"
            print(error_msg)
            return error_msg

## 处理费用

定义一个异步函数，通过创建和注册必要的代理，然后调用它们来处理费用。
- 该函数通过加载环境变量、创建必要的代理并将其注册为插件来处理费用。
- 它会创建一个包含两个代理的群聊，并发送提示消息，根据费用数据生成电子邮件和饼图。
- 它会处理在聊天调用过程中发生的任何错误，并确保正确清理代理。


In [ ]:
# 处理费用的主函数 - 就像整个报销流程的指挥官
async def process_expenses():
    """处理报销费用的主流程"""
    # 加载环境变量（API密钥等）
    load_dotenv()
    
    # 设置小型模型（用于OCR）的执行参数
    settings_slm = OpenAIChatPromptExecutionSettings(service_id="gpt-4o")
    
    # 设置大型模型（用于邮件生成）的执行参数
    settings_llm = OpenAIChatPromptExecutionSettings(service_id="open_ai")
    
    # 创建OCR代理 - 专门负责从图片中提取数据
    ocr_agent = ChatCompletionAgent(
        kernel=_create_kernel_with_chat_completion("ocrAgent"),  # 创建专用Kernel
        name="ocr_agent",  # 代理名称
        instructions="Extract travel expense data from the receipt image in the prompt using the 'extract_text' function from the 'ocrAgent' plugin. Return the data in the format 'date|description|amount|category' separated by semicolons.",
        arguments=KernelArguments(settings=settings_slm) # 设置执行参数
    )
    
    # 创建邮件代理 - 专门负责生成报销邮件
    email_agent = ChatCompletionAgent(
            kernel=_create_kernel_with_chat_completion("expenseEmailAgent"),
            name="email_agent",
            instructions="Take the travel expense data from the previous agent and generate a professional expense claim email using the 'generate_expense_email' function from the 'expenseEmailAgent' plugin, then pass the data forward.",
            arguments=KernelArguments(settings=settings_llm)
        )

    # 创建一个"厨房"（Kernel），用于管理所有AI处理
    kernel = Kernel()

    # Use fixed path to receipt.jpg in the same folder
    # 指定收据图片的路径（假设图片在当前目录）
    image_path = "./receipt.jpg"
    
    # Create a structured message with text and image content for OCR processing
    image_url_str = f"file://{image_path}"
    
    # Using the correct format for multi-modal content
    # 创建一个包含文本和图片的用户消息
    user_message = ChatMessageContent(
        role=AuthorRole.USER,
        items=[
            TextContent(text="""
            Please extract the raw text from this receipt image, focusing on travel expenses like dates, descriptions, amounts, and categories (e.g., Transportation, Accommodation, Meals, Miscellaneous).
            Then generate a professional expense claim email.
                        """),  # 请从此收据图片中提取原始文本，重点关注差旅费用，如日期、描述、金额和类别（如交通、住宿、餐饮、杂项）。然后生成一份专业的报销申请邮件。
            ImageContent.from_image_file(path=image_path)
        ]
    )

    # Register plugins with the kernel
    # 将插件注册到Kernel
    kernel.add_plugin(OCRAgentPlugin(), plugin_name="ocrAgent")
    kernel.add_plugin(ExpenseEmailAgent(), plugin_name="expenseEmailAgent")

    # Create group chat
    # 创建代理群组聊天 - 让两个代理协同工作
    chat = AgentGroupChat(
        agents=[ocr_agent, email_agent],
        selection_strategy=SequentialSelectionStrategy(initial_agent=ocr_agent),
        termination_strategy=DefaultTerminationStrategy(maximum_iterations=1)
    )

    # Add user message with prompt
    # 将用户消息添加到聊天中
    await chat.add_chat_message(user_message)
    print(f"# User message added to chat with receipt image")

    # 让代理开始工作，并逐个显示结果
    async for content in chat.invoke():
        print(f"# Agent - {content.name or '*'}: '{content.content}'")


## 主函数

定义主函数以清空控制台并异步运行 `process_expenses` 函数。


In [8]:
async def main():
    """主函数：清空控制台并运行报销处理流程"""
    # 清空控制台（Windows用'cls'，其他系统用'clear'）
    # Clear the console
    os.system('cls' if os.name=='nt' else 'clear')

    # Run the async agent code
    await process_expenses()

await main()

# User message added to chat with receipt image
# Agent - ocr_agent: 'Extracted travel expense data:  
2 May '22|Lunch (Penne Alfredo, Chicken Wrap, Jungle Colada, Melon Lime Cooler)|75.15|Meals  

Below is a professional expense claim email:

---

**Subject:** Expense Claim Submission

**Dear [Recipient's Name],**

I hope this email finds you well. Please find below the details of a travel-related expense I incurred during my recent trip. I kindly request reimbursement as per company policy:

- **Date:** 2nd May 2022  
- **Description:** Lunch expense (Penne Alfredo, Chicken Wrap, Jungle Colada, Melon Lime Cooler)  
- **Amount:** $75.15  
- **Category:** Meals  

Attached to this email is the receipt for your reference. Please let me know if you need any further information or clarification regarding this claim.

Thank you for your prompt attention to this matter.

**Best regards,**  
[Your Full Name]  
[Your Contact Information]  

---'



---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
